# AcuDock Scout

**Smart virtual screening -- find top hits by docking a fraction of your library.**

Traditional virtual screening docks **every** compound in a library. For large libraries
(thousands to millions of molecules), this can take days. AcuDock Scout uses
**active learning** to intelligently select which molecules to dock next, typically
finding >90% of the top hits while docking less than 10% of the library.

### How It Works

1. **Bootstrap** -- Dock a small random sample to give the ML model its first training data
2. **Learn** -- Train a fast model (Random Forest) to predict scores from molecular structure
3. **Select** -- The model picks the most promising undocked molecules, balancing
   predicted good binders with uncertain compounds it wants to learn more about
4. **Dock** -- Only dock the selected batch
5. **Repeat** -- Each cycle, the model gets smarter and the hits get better

Based on the HASTEN approach (Graff et al., 2021).

### Getting Started

1. Click **Runtime > Run all** to run all cells
2. The first cell installs dependencies (~2-3 min) and the runtime will **automatically restart**
3. After restart, click **Runtime > Run all** again
4. Configure your campaign in the widget panel and click **Start Campaign**

### Built-in Libraries

- Type **`demo`** for ~500 common drug variants (Aspirin, Ibuprofen, Caffeine, etc.)
- Type **`acyl-thiourea`** for ~500 acyl thiourea compounds (R-C(=O)-NH-C(=S)-NH-R')
- Or paste your own `Name,SMILES` pairs

---

*License: MIT | Platform: Google Colab | Engine: AutoDock Vina (CPU) or Uni-Dock (GPU)*


In [ ]:
#@title Step 1: Install Dependencies (run once, then runtime restarts)
# === Step 1: Install Dependencies ===
!pip install -q vina meeko gemmi rdkit prody py3Dmol openbabel-wheel pdbfixer pandas numpy scipy scikit-learn matplotlib seaborn ipywidgets reportlab prolif

# Clone repo
!git clone https://github.com/Grimlock5310/AcuDock.git /content/AcuDock 2>/dev/null || (cd /content/AcuDock && git pull)

# Install Uni-Dock for GPU-accelerated docking (falls back to Vina if no GPU)
# Uncomment the next line for 1000x+ speedup on NVIDIA GPUs (compute capability >= 7.0):
!wget -q https://github.com/dptech-corp/Uni-Dock/releases/download/1.1.0/unidock-1.1.0-cuda120-linux-x86_64 -O /usr/local/bin/unidock && chmod +x /usr/local/bin/unidock && echo "Uni-Dock GPU v1.1.0 installed" || echo "Uni-Dock download failed"

import os
os.kill(os.getpid(), 9)

## Active Learning Screening Interface

The cells below load the screening tools and display the campaign interface.
Configure your protein target, active learning parameters, and compound library,
then click **Start Campaign**. Results include convergence plots, score distributions,
a ranked hit list, and a 3D viewer of the top hit.


In [ ]:
#@title Step 2: Load Libraries
# === Step 2: Imports ===
import warnings
warnings.filterwarnings('ignore')

import os, sys, io, time, random
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

sys.path.insert(0, '/content/AcuDock')
import acudock_utils as utils
from acudock_surrogate import SurrogateModel
from acudock_screening import BatchDockingManager, cluster_hits, compute_tanimoto_matrix

WORK_DIR = '/content/acudock_scout'
os.makedirs(WORK_DIR, exist_ok=True)

print('AcuDock Scout loaded.')
print(utils.get_docking_engine_status())
try:
    import acudock_report as report
except ImportError:
    report = None
    print('Note: reportlab not installed. PDF reports unavailable.')


In [ ]:
#@title Step 3: Launch Interactive Interface
# === Step 3: Launch Interactive Interface ===
import ipywidgets as widgets
from IPython.display import display, HTML, FileLink, clear_output
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')


def generate_demo_library():
    """Generate a demo compound library of ~500 SMILES."""
    seed_drugs = [
        ('Aspirin', 'CC(=O)Oc1ccccc1C(=O)O'),
        ('Ibuprofen', 'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
        ('Caffeine', 'Cn1c(=O)c2c(ncn2C)n(C)c1=O'),
        ('Acetaminophen', 'CC(=O)Nc1ccc(O)cc1'),
        ('Naproxen', 'COc1ccc2cc(ccc2c1)C(C)C(=O)O'),
        ('Metformin', 'CN(C)C(=N)NC(=N)N'),
        ('Omeprazole', 'COc1ccc2[nH]c(nc2c1)S(=O)Cc1ncc(C)c(OC)c1C'),
        ('Atorvastatin', 'CC(C)c1n(CC[C@@H](O)C[C@@H](O)CC(=O)O)c(c2ccc(F)cc2)c(c1c1ccccc1)C(=O)Nc1ccccc1'),
        ('Celecoxib', 'Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)cc1'),
        ('Diclofenac', 'OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl'),
        ('Ciprofloxacin', 'O=C(O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O'),
        ('Loratadine', 'CCOC(=O)N1CCC(=C2c3ccc(Cl)cc3CCc3cccnc32)CC1'),
        ('Sildenafil', 'CCCc1nn(C)c2c1nc(nc2OCC)c1cc(ccc1OCC)S(=O)(=O)N1CCN(C)CC1'),
        ('Warfarin', 'CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O'),
        ('Fluoxetine', 'CNCCC(Oc1ccc(C(F)(F)F)cc1)c1ccccc1'),
        ('Tamoxifen', 'CCC(=C(c1ccccc1)c1ccc(OCCN(C)C)cc1)c1ccccc1'),
        ('Metoprolol', 'COCCc1ccc(OCC(O)CNC(C)C)cc1'),
        ('Losartan', 'CCCCc1nc(Cl)c(n1Cc1ccc(-c2ccccc2-c2nnn[nH]2)cc1)CO'),
        ('Amlodipine', 'CCOC(=O)C1=C(COCCN)NC(C)=C(C1c1ccccc1Cl)C(=O)OC'),
        ('Captopril', 'CC(CS)C(=O)N1CCCC1C(=O)O'),
        ('Furosemide', 'NS(=O)(=O)c1cc(C(=O)O)c(NCc2ccco2)cc1Cl'),
        ('Clopidogrel', 'COC(=O)C(c1ccccc1Cl)N1CCc2sccc2C1'),
        ('Propranolol', 'CC(C)NCC(O)COc1cccc2ccccc12'),
        ('Lidocaine', 'CCN(CC)CC(=O)Nc1c(C)cccc1C'),
        ('Trimethoprim', 'COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC'),
        ('Chloroquine', 'CCN(CC)CCCC(C)Nc1ccnc2cc(Cl)ccc12'),
        ('Phenytoin', 'O=C1NC(=O)C(c2ccccc2)(c2ccccc2)N1'),
        ('Carbamazepine', 'NC(=O)N1c2ccccc2C=Cc2ccccc21'),
        ('Fluconazole', 'OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F'),
        ('Riluzole', 'Nc1nc2ccc(OC(F)(F)F)cc2s1'),
        ('Sorafenib', 'CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1'),
        ('Erlotinib', 'C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1'),
        ('Diphenhydramine', 'CN(C)CCOC(c1ccccc1)c1ccccc1'),
    ]

    RDLogger.DisableLog('rdApp.*')
    library = list(seed_drugs)
    random.seed(42)

    for name, smi in seed_drugs:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        for _ in range(15):
            if len(library) >= 500:
                break
            try:
                order = list(range(mol.GetNumAtoms()))
                random.shuffle(order)
                renumbered = Chem.RenumberAtoms(mol, order)
                canon = Chem.MolToSmiles(Chem.MolFromSmiles(Chem.MolToSmiles(renumbered, canonical=False)))
                if canon and Chem.MolFromSmiles(canon) and canon not in [s for _, s in library]:
                    library.append((f'{name}_v{len(library)}', canon))
            except Exception:
                continue

    scaffolds = ['c1ccccc1', 'c1ccncc1', 'c1ccc2ccccc2c1', 'C1CCCCC1',
                 'c1ccoc1', 'c1ccsc1', 'c1ccc2nccnc2c1', 'c1ccc2[nH]ccc2c1']
    subs = ['O', 'N', 'C(=O)O', 'F', 'Cl', 'OC', 'NC', 'C(F)(F)F',
            'C(=O)N', 'S(=O)(=O)N', 'CC(=O)O', 'OCC', 'C#N']
    for sc in scaffolds:
        for sub in subs:
            if len(library) >= 500:
                break
            test = f'{sc}{sub}'
            mol = Chem.MolFromSmiles(test)
            if mol:
                canon = Chem.MolToSmiles(mol)
                if canon not in [s for _, s in library]:
                    library.append((f'Frag_{len(library)}', canon))

    return library


def generate_acyl_thiourea_library():
    """Generate a library of ~500 acyl thiourea compounds.

    Acyl thioureas (R-C(=O)-NH-C(=S)-NH-R\) have diverse
    biological activities including antimicrobial, anticancer,
    and enzyme inhibition. This library systematically
    combines acyl groups with amine substituents to explore
    the structure-activity landscape.
    """
    RDLogger.DisableLog('rdApp.*')
    library = []

    # Core acyl thiourea scaffold: R-C(=O)-NC(=S)N-R2
    # Acyl chloride side (R-C=O)
    acyl_groups = [
        ('Benzoyl', 'O=C(c1ccccc1)'),
        ('4ClBenzoyl', 'O=C(c1ccc(Cl)cc1)'),
        ('4BrBenzoyl', 'O=C(c1ccc(Br)cc1)'),
        ('4FBenzoyl', 'O=C(c1ccc(F)cc1)'),
        ('4MeBenzoyl', 'O=C(c1ccc(C)cc1)'),
        ('4OMeBenzoyl', 'O=C(c1ccc(OC)cc1)'),
        ('4NO2Benzoyl', 'O=C(c1ccc([N+](=O)[O-])cc1)'),
        ('4OHBenzoyl', 'O=C(c1ccc(O)cc1)'),
        ('3ClBenzoyl', 'O=C(c1cccc(Cl)c1)'),
        ('2ClBenzoyl', 'O=C(c1ccccc1Cl)'),
        ('35diClBenzoyl', 'O=C(c1cc(Cl)cc(Cl)c1)'),
        ('24diClBenzoyl', 'O=C(c1ccc(Cl)cc1Cl)'),
        ('4CF3Benzoyl', 'O=C(c1ccc(C(F)(F)F)cc1)'),
        ('Naphthoyl', 'O=C(c1ccc2ccccc2c1)'),
        ('Furoyl', 'O=C(c1ccoc1)'),
        ('Thiophenoyl', 'O=C(c1ccsc1)'),
        ('Nicotinoyl', 'O=C(c1cccnc1)'),
        ('Isonicotinoyl', 'O=C(c1ccncc1)'),
        ('Cinnamoyl', 'O=C(/C=C/c1ccccc1)'),
        ('Acetyl', 'O=C(C)'),
        ('Propanoyl', 'O=C(CC)'),
        ('Pivaloyl', 'O=C(C(C)(C)C)'),
        ('Cyclohexanoyl', 'O=C(C1CCCCC1)'),
        ('Phenylacetyl', 'O=C(Cc1ccccc1)'),
    ]

    # Amine substituents on thiourea nitrogen (R2)
    amines = [
        ('Ph', 'c1ccccc1'),
        ('4ClPh', 'c1ccc(Cl)cc1'),
        ('4BrPh', 'c1ccc(Br)cc1'),
        ('4FPh', 'c1ccc(F)cc1'),
        ('4MePh', 'c1ccc(C)cc1'),
        ('4OMePh', 'c1ccc(OC)cc1'),
        ('4NO2Ph', 'c1ccc([N+](=O)[O-])cc1'),
        ('3CF3Ph', 'c1cccc(C(F)(F)F)c1'),
        ('2Naphthyl', 'c1ccc2cc(ccc2c1)'),
        ('Benzyl', 'Cc1ccccc1'),
        ('4ClBenzyl', 'Cc1ccc(Cl)cc1'),
        ('Cyclohexyl', 'C1CCCCC1'),
        ('Ethyl', 'CC'),
        ('nButyl', 'CCCC'),
        ('Allyl', 'CC=C'),
        ('Morpholinyl', 'N1CCOCC1'),
        ('Piperidinyl', 'N1CCCCC1'),
        ('Piperazinyl', 'N1CCNCC1'),
        ('2Pyridyl', 'c1ccncc1'),
        ('2Thiazolyl', 'c1nccs1'),
    ]

    # Build acyl thiourea: acyl-NH-C(=S)-NH-amine
    # SMARTS pattern: R-C(=O)NC(=S)N-R2
    for acyl_name, acyl_smi in acyl_groups:
        for amine_name, amine_smi in amines:
            if len(library) >= 500:
                break
            smi = f'{acyl_smi}NC(=S)N{amine_smi}'
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                canon = Chem.MolToSmiles(mol)
                name = f'{acyl_name}_{amine_name}'
                if canon not in [s for _, s in library]:
                    library.append((name, canon))
        if len(library) >= 500:
            break

    # Add metal-coordinating variants (common in thiourea complexes)
    # These add chelating groups that coordinate metals like Cu, Ni, Zn
    chelating_acyls = [
        ('Salicyloyl', 'O=C(c1ccccc1O)'),
        ('2PyridylCO', 'O=C(c1ccccn1)'),
        ('AnthraniloylOMe', 'O=C(c1ccccc1NC(=O)OC)'),
        ('2OHNaphthoyl', 'O=C(c1cc(O)c2ccccc2c1)'),
        ('QuinolineCO', 'O=C(c1ccc2ncccc2c1)'),
    ]
    for acyl_name, acyl_smi in chelating_acyls:
        for amine_name, amine_smi in amines[:10]:
            if len(library) >= 500:
                break
            smi = f'{acyl_smi}NC(=S)N{amine_smi}'
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                canon = Chem.MolToSmiles(mol)
                name = f'{acyl_name}_{amine_name}_chelate'
                if canon not in [s for _, s in library]:
                    library.append((name, canon))
        if len(library) >= 500:
            break

    random.seed(42)
    random.shuffle(library)
    return library


def run_scout_campaign(pdb_id, residues_str, box_size, bootstrap_size,
                        batch_size, n_cycles, ucb_beta, exhaustiveness,
                        engine, library_text, output_area):
    """Run full active learning campaign."""
    with output_area:
        clear_output(wait=True)
        print('Starting AcuDock Scout campaign...')

    try:
        pdb_id = pdb_id.strip().upper()
        bootstrap_size = int(bootstrap_size)
        batch_size = int(batch_size)
        n_cycles = int(n_cycles)
        exhaustiveness = int(exhaustiveness)
        ucb_beta = float(ucb_beta)

        lib_choice = library_text.strip().lower()
        if lib_choice in ('demo', 'demo-drugs', '') or not lib_choice:
            with output_area:
                print('Generating drug demo library (~500 compounds)...')
            compound_library = generate_demo_library()
        elif lib_choice in ('acyl-thiourea', 'thiourea', 'acyl thiourea'):
            with output_area:
                print('Generating acyl thiourea library (~500 compounds)...')
            compound_library = generate_acyl_thiourea_library()
        else:
            compound_library = []
            for line in library_text.strip().split('\n'):
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = line.split(',', 1) if ',' in line else line.split('\t', 1)
                if len(parts) == 2:
                    name, smi = parts[0].strip(), parts[1].strip()
                else:
                    smi = parts[0].strip()
                    name = f'Cpd_{len(compound_library)+1}'
                if Chem.MolFromSmiles(smi):
                    compound_library.append((name, smi))

        if len(compound_library) < 20:
            with output_area:
                print('Error: Need at least 20 compounds.')
            return

        with output_area:
            print(f'Library: {len(compound_library)} compounds')
            print(f'Preparing protein {pdb_id}...')

        protein_pdb = utils.prepare_protein(pdb_id, output_dir=WORK_DIR)
        protein_pdbqt = utils.pdb_to_pdbqt(protein_pdb)

        residues = None
        if residues_str.strip():
            residues = [int(r.strip()) for r in residues_str.split(',') if r.strip().isdigit()]
        if residues:
            center = utils.get_binding_site_center(protein_pdb, chain='A', residues=residues)
        else:
            try:
                site_info = utils.detect_binding_site(pdb_id, output_dir=WORK_DIR)
                center = site_info['center']
                with output_area:
                    print(f'  Auto-detected binding site near {site_info.get("het_name","?")} ({site_info["method"]})')
            except Exception:
                center = utils.get_binding_site_center(protein_pdb, chain='A', residues=None)
        box = [int(box_size)] * 3
        with output_area:
            print(f'  Center: [{center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f}]')

        use_unidock = 'Uni-Dock' in engine
        if use_unidock:
            from acudock_utils import check_unidock_available
            if not check_unidock_available():
                with output_area:
                    print('Uni-Dock not found, falling back to Vina.')
                use_unidock = False
        manager = BatchDockingManager(
            receptor_pdbqt=protein_pdbqt, center=center, box_size=box,
            exhaustiveness=exhaustiveness, n_poses=5, output_dir=WORK_DIR,
            use_unidock=use_unidock
        )
        library = manager.load_library(compound_library, shuffle=True)
        surrogate = SurrogateModel(fp_radius=2, fp_bits=2048, n_estimators=200)
        cycle_metrics = []

        # Bootstrap
        with output_area:
            print(f'\n=== CYCLE 0: BOOTSTRAP ({bootstrap_size} compounds) ===')
            print('  Docking random sample (this may take several minutes)...')
        bootstrap = manager.random_sample(library, bootstrap_size)
        t0 = time.time()
        manager.dock_batch(bootstrap)
        bt = time.time() - t0
        stats = manager.get_statistics()
        with output_area:
            print(f'  Docked: {stats["total_docked"]} in {bt:.0f}s')
            print(f'  Best: {stats["best_score"]} kcal/mol')

        results_df = manager.get_all_results()
        metrics = surrogate.train(results_df['SMILES'].tolist(), results_df['Best_Score'].values)
        with output_area:
            print(f'  Surrogate R2={metrics["train_r2"]}, RMSE={metrics["train_rmse"]}')
        cycle_metrics.append({
            'cycle': 0, 'n_docked': stats['total_docked'],
            'pct_library': stats['total_docked'] / len(library) * 100,
            'best_score': stats['best_score'],
            'train_r2': metrics['train_r2'], 'cv_r2': metrics['cv_r2_mean'],
            'train_rmse': metrics['train_rmse'],
            'n_strong': stats.get('n_strong_hits', 0),
            'n_moderate': stats.get('n_moderate_hits', 0),
        })

        # AL Cycles
        for cycle in range(1, n_cycles + 1):
            with output_area:
                print(f'\n=== CYCLE {cycle}/{n_cycles}: ACTIVE LEARNING ===')

            candidates = [smi for _, smi in library if smi not in manager.docked_smiles]
            if not candidates:
                with output_area:
                    print('All compounds docked. Stopping.')
                break

            selected = surrogate.select_next_batch(
                candidates, batch_size=min(batch_size, len(candidates)), beta=ucb_beta
            )
            smi_to_name = {smi: name for name, smi in library}
            batch_compounds = [(smi_to_name.get(smi, f'AL_{cycle}_{i}'), smi)
                               for i, (_, smi, _) in enumerate(selected)]

            with output_area:
                print(f'  Selected {len(batch_compounds)} compounds via UCB')
                print(f'  Docking batch...')
            t0 = time.time()
            manager.dock_batch(batch_compounds)

            results_df = manager.get_all_results()
            metrics = surrogate.train(results_df['SMILES'].tolist(), results_df['Best_Score'].values)
            stats = manager.get_statistics()
            pct_done = stats['total_docked'] / len(library) * 100

            with output_area:
                print(f'  Docked: {stats["total_docked"]}/{len(library)} ({pct_done:.1f}%)')
                print(f'  Best: {stats["best_score"]} kcal/mol | R2={metrics["train_r2"]}')

            cycle_metrics.append({
                'cycle': cycle, 'n_docked': stats['total_docked'],
                'pct_library': pct_done, 'best_score': stats['best_score'],
                'train_r2': metrics['train_r2'], 'cv_r2': metrics['cv_r2_mean'],
                'train_rmse': metrics['train_rmse'],
                'n_strong': stats.get('n_strong_hits', 0),
                'n_moderate': stats.get('n_moderate_hits', 0),
            })

        # Results
        with output_area:
            print(f'\n=== BUILDING RESULTS ===')

        final_stats = manager.get_statistics()
        top_hits = manager.get_top_hits(n=20)
        extra_cols = ['Name', 'SMILES', 'Best_Score', 'MW', 'LogP']
        for col in ['QED', 'SA_Score', 'PAINS_Count', 'LE']:
            if col in top_hits.columns:
                extra_cols.append(col)
        display_df = top_hits[[c for c in extra_cols if c in top_hits.columns]].copy()
        display_df.insert(0, 'Rank', range(1, len(display_df) + 1))
        display_df['SMILES'] = display_df['SMILES'].str[:40] + '...'

        metrics_df = pd.DataFrame(cycle_metrics)

        with output_area:
            print('Generating plots...')

        # Convergence
        fig1, ax1 = plt.subplots(figsize=(8, 5))
        ax1.plot(metrics_df['pct_library'], metrics_df['best_score'],
                 'o-', color='#2196F3', linewidth=2, markersize=8, label='Best Score')
        ax1.set_xlabel('% Library Docked')
        ax1.set_ylabel('Best Score (kcal/mol)')
        ax1.set_title('Active Learning Convergence')
        ax1.invert_yaxis()
        ax1.axvline(x=10, color='red', linestyle=':', alpha=0.7, label='10% HASTEN target')
        ax1.legend()
        plt.tight_layout()
        conv_path = os.path.join(WORK_DIR, 'convergence.png')
        fig1.savefig(conv_path, dpi=120, bbox_inches='tight')
        plt.close(fig1)

        # Score distribution
        all_results = manager.get_all_results()
        bs_scores = all_results.head(bootstrap_size)['Best_Score'].dropna()
        al_scores = all_results.iloc[bootstrap_size:]['Best_Score'].dropna()

        fig2, ax2 = plt.subplots(figsize=(8, 5))
        if len(bs_scores) > 1:
            sns.kdeplot(data=bs_scores, ax=ax2, color='#90CAF9', fill=True,
                        alpha=0.4, label=f'Bootstrap (n={len(bs_scores)})')
        if len(al_scores) > 1:
            sns.kdeplot(data=al_scores, ax=ax2, color='#EF5350', fill=True,
                        alpha=0.4, label=f'AL-Selected (n={len(al_scores)})')
        ax2.axvline(x=-7, color='orange', linestyle='--', alpha=0.8, label='Moderate')
        ax2.axvline(x=-8, color='red', linestyle='--', alpha=0.8, label='Strong')
        ax2.set_xlabel('Vina Score (kcal/mol)')
        ax2.set_ylabel('Density')
        ax2.set_title('Score Distribution: Bootstrap vs AL')
        ax2.legend()
        plt.tight_layout()
        dist_path = os.path.join(WORK_DIR, 'distribution.png')
        fig2.savefig(dist_path, dpi=120, bbox_inches='tight')
        plt.close(fig2)

        # Surrogate performance
        fig3, ax3 = plt.subplots(figsize=(8, 5))
        ax3.plot(metrics_df['cycle'], metrics_df['train_r2'],
                 'o-', color='#2196F3', label='Train R2')
        ax3.plot(metrics_df['cycle'], metrics_df['cv_r2'],
                 's--', color='#FF9800', label='CV R2')
        ax3.set_xlabel('Cycle')
        ax3.set_ylabel('R2')
        ax3.set_title('Surrogate Model Performance')
        ax3.legend()
        ax3.set_ylim(-0.1, 1.1)
        plt.tight_layout()
        surr_path = os.path.join(WORK_DIR, 'surrogate.png')
        fig3.savefig(surr_path, dpi=120, bbox_inches='tight')
        plt.close(fig3)

        # 3D viewer of top hit
        with output_area:
            print('Visualizing top hit...')
        best_hit = top_hits.iloc[0]
        best_viewer_paths = None
        try:
            best_smi = best_hit['SMILES']
            lig_pdbqt, _ = utils.prepare_ligand(best_smi, name='best_hit', output_dir=WORK_DIR)
            _, en, poses_path = utils.run_vina(
                protein_pdbqt, lig_pdbqt,
                center=center, box_size=box,
                exhaustiveness=32, n_poses=5
            )
            best_viewer_paths = (protein_pdb, poses_path)
        except Exception as e:
            with output_area:
                print(f'Visualization failed: {e}')

        # Save CSVs
        results_csv = os.path.join(WORK_DIR, 'all_results.csv')
        all_results.to_csv(results_csv, index=False)

        with output_area:
            clear_output(wait=True)
            print(f'=== CAMPAIGN COMPLETE ===')
            print(f'Total docked: {final_stats["total_docked"]}/{len(library)} '
                  f'({final_stats["total_docked"]/len(library)*100:.1f}%)')
            print(f'Best score: {final_stats["best_score"]} kcal/mol')
            print(f'Strong hits (<-8): {final_stats.get("n_strong_hits", 0)}')
            print(f'Moderate hits (<-7): {final_stats.get("n_moderate_hits", 0)}')
            print()
            display(HTML('<h4>Top 20 Hits</h4>'))
            display(display_df)
            print()
            display(HTML('<h4>Cycle Metrics</h4>'))
            display(metrics_df)
            print()
            display(HTML('<h4>Convergence</h4>'))
            from IPython.display import Image as IPyImage
            display(IPyImage(filename=conv_path))
            print()
            display(HTML('<h4>Score Distribution</h4>'))
            display(IPyImage(filename=dist_path))
            print()
            display(HTML('<h4>Surrogate Model</h4>'))
            display(IPyImage(filename=surr_path))
            display(FileLink(results_csv, result_html_prefix='Download All Results: '))
            if best_viewer_paths:
                print()
                display(HTML('<h4>Top Hit 3D Viewer</h4>'))
                _v_out = widgets.Output()
                display(_v_out)
                with _v_out:
                    view = utils.visualize_pose(best_viewer_paths[0], best_viewer_paths[1], pose_index=0)
                    view.show()

                # 2D Interaction Diagram
                try:
                    display(HTML('<h4>2D Interaction Diagram</h4>'))
                    _diag_path = os.path.join(WORK_DIR, 'top_hit_interaction_2d.png')
                    _top_smiles = top_hits.iloc[0]['SMILES'] if 'SMILES' in top_hits.columns else None
                    if _top_smiles:
                        utils.generate_interaction_diagram_2d(
                            best_viewer_paths[0], best_viewer_paths[1], _top_smiles,
                            pose_index=0, output_path=_diag_path)
                        from IPython.display import Image as IPyImage
                        display(IPyImage(filename=_diag_path))
                except Exception as _e2d:
                    print(f'2D diagram: {_e2d}')

                # Blender Export
                try:
                    _top_name = top_hits.iloc[0].get('Name', 'top_hit') if not top_hits.empty else 'top_hit'
                    _zip = utils.export_pdb_bundle(best_viewer_paths[0], best_viewer_paths[1],
                                                   ligand_name=str(_top_name))
                    utils.create_download_link(_zip, 'Download PDB Bundle (Blender)')
                except Exception as _ebl:
                    print(f'Blender export: {_ebl}')

            # PDF Report button
            if report is not None:
                print()
                _spdf_btn = widgets.Button(description='Download PDF Report',
                                           button_style='info', icon='file-pdf-o',
                                           layout=widgets.Layout(width='220px'))
                _spdf_out = widgets.Output()
                def _on_spdf(btn, _pdb=pdb_id, _metrics=cycle_metrics,
                             _top=top_hits, _plots={'convergence': conv_path,
                             'distribution': dist_path, 'surrogate': surr_path},
                             _bvp=best_viewer_paths):
                    btn.disabled = True
                    btn.description = 'Generating...'
                    try:
                        config = {'pdb_id': _pdb, 'library_size': len(library),
                                  'bootstrap_size': bootstrap_size, 'batch_size': batch_size,
                                  'n_cycles': n_cycles, 'ucb_beta': ucb_beta,
                                  'exhaustiveness': exhaustiveness}
                        pdf_path = os.path.join(WORK_DIR, f'scout_{_pdb}_report.pdf')
                        report.generate_scout_pdf(pdf_path, config, _top, plots_dict=_plots)
                        with _spdf_out:
                            clear_output(wait=True)
                            display(FileLink(pdf_path, result_html_prefix='PDF Ready: '))
                    except Exception as ex:
                        with _spdf_out:
                            print(f'PDF error: {ex}')
                    finally:
                        btn.disabled = False
                        btn.description = 'Download PDF Report'
                _spdf_btn.on_click(_on_spdf)
                display(_spdf_btn)
                display(_spdf_out)

            # Google Drive auto-save
            if sc_drive_save.value:
                try:
                    drive_dir = utils.mount_google_drive()
                    if drive_dir:
                        save_dir = utils.ensure_drive_dir(os.path.join(drive_dir, 'AcuDock'))
                        import shutil
                        shutil.copy2(results_csv, os.path.join(save_dir, f'scout_{pdb_id}_results.csv'))
                        pd.DataFrame(cycle_metrics).to_csv(
                            os.path.join(save_dir, f'scout_{pdb_id}_metrics.csv'), index=False)
                        for p in [conv_path, dist_path, surr_path]:
                            if os.path.exists(p):
                                shutil.copy2(p, save_dir)
                        pdf_drive = os.path.join(WORK_DIR, f'scout_{pdb_id}_report.pdf')
                        if os.path.exists(pdf_drive):
                            shutil.copy2(pdf_drive, save_dir)
                        print(f'\nResults saved to Google Drive: {save_dir}')
                except Exception as drive_err:
                    print(f'\nDrive save failed: {drive_err}')

    except Exception as e:
        with output_area:
            print(f'\nERROR: {str(e)}')
            import traceback
            traceback.print_exc()


# ---------------------------------------------------------------------------
# Build ipywidgets Interface
# ---------------------------------------------------------------------------

style = {'description_width': '180px'}
layout_input = widgets.Layout(width='95%')

# === Campaign Tab ===
sc_pdb = widgets.Text(value='1HSG', description='PDB ID:', style=style, layout=layout_input)
sc_residues = widgets.Text(value='23,24,25,26,27,28,29,30',
                            description='Active Site Residues:', style=style, layout=layout_input)
sc_box = widgets.IntSlider(value=20, min=15, max=40, step=5,
                            description='Box Size (A):', style=style, layout=layout_input)
sc_bootstrap = widgets.IntSlider(value=100, min=50, max=2000, step=50,
                                  description='Bootstrap Size:', style=style, layout=layout_input)
sc_batch = widgets.IntSlider(value=50, min=25, max=1000, step=25,
                              description='Batch Size:', style=style, layout=layout_input)
sc_cycles = widgets.IntSlider(value=3, min=1, max=20, step=1,
                               description='AL Cycles:', style=style, layout=layout_input)
sc_beta = widgets.FloatSlider(value=1.5, min=0.5, max=3.0, step=0.1,
                               description='UCB Beta:', style=style, layout=layout_input)
sc_exhaust = widgets.IntSlider(value=8, min=4, max=32, step=4,
                                description='Exhaustiveness:', style=style, layout=layout_input)
sc_engine = widgets.Dropdown(options=['Vina (CPU)', 'Uni-Dock (GPU)'],
                             value='Vina (CPU)', description='Engine:', style=style,
                             layout=layout_input)
sc_library = widgets.Textarea(
    value='demo', description='Library:', style=style,
    layout=widgets.Layout(width='95%', height='80px'),
    placeholder='"demo" = common drugs, "acyl-thiourea" = acyl thiourea series, or paste Name,SMILES lines'
)
sc_drive_save = widgets.Checkbox(value=False, description='Auto-save to Google Drive',
                                 style=style, layout=layout_input)
sc_btn = widgets.Button(description='Start Campaign', button_style='primary',
                         layout=widgets.Layout(width='95%', height='40px'))
sc_output = widgets.Output(layout=widgets.Layout(width='100%', min_height='400px',
                                                  border='1px solid #ddd'))

def on_campaign(btn):
    btn.disabled = True
    btn.description = 'Running campaign...'
    try:
        run_scout_campaign(sc_pdb.value, sc_residues.value, sc_box.value,
                           sc_bootstrap.value, sc_batch.value, sc_cycles.value,
                           sc_beta.value, sc_exhaust.value, sc_engine.value,
                           sc_library.value, sc_output)
    finally:
        btn.disabled = False
        btn.description = 'Start Campaign'

sc_btn.on_click(on_campaign)

campaign_tab = widgets.HBox([
    widgets.VBox([
        widgets.HTML('<h3>Target Protein</h3>'),
        sc_pdb, sc_residues, sc_box,
        widgets.HTML('<h3>Active Learning</h3>'),
        sc_bootstrap, sc_batch, sc_cycles, sc_beta, sc_exhaust, sc_engine,
        widgets.HTML('<h3>Compound Library</h3>'),
        sc_library,
        widgets.HTML('<h3>Options</h3>'),
        sc_drive_save,
        sc_btn,
    ], layout=widgets.Layout(width='40%', padding='10px')),
    widgets.VBox([sc_output],
                 layout=widgets.Layout(width='60%', padding='10px'))
])

# === About Tab ===
about_tab = widgets.HTML(value="""
<div style="max-width:800px; padding:20px; font-family:sans-serif;">
<h3>AcuDock Scout — Active Learning Virtual Screening</h3>

<h4>Algorithm</h4>
<ol>
<li><b>Bootstrap:</b> Dock random 2% of library to seed ML model</li>
<li><b>Train:</b> Random Forest on Morgan fingerprints + descriptors</li>
<li><b>Select:</b> UCB acquisition (exploit predicted good + explore uncertain)</li>
<li><b>Dock:</b> Dock selected batch with Vina or Uni-Dock</li>
<li><b>Repeat</b> until convergence or cycle limit</li>
</ol>

<h4>Key Parameters</h4>
<table border="1" cellpadding="5" style="border-collapse:collapse;">
<tr><th>Parameter</th><th>Description</th></tr>
<tr><td>Bootstrap Size</td><td>Initial random sample (2-5% of library)</td></tr>
<tr><td>Batch Size</td><td>Compounds per AL cycle</td></tr>
<tr><td>UCB Beta</td><td>Higher = more exploration, lower = more exploitation</td></tr>
<tr><td>Exhaustiveness</td><td>Vina search thoroughness (8=fast, 32=thorough)</td></tr>
</table>

<h4>References</h4>
<ul>
<li>HASTEN: Graff et al., <i>Chem. Sci.</i>, 2021</li>
<li>&gt;90% of top hits found with &lt;10% docking effort</li>
</ul>

<h4>Engines</h4>
<ul>
<li><b>Vina (CPU):</b> Works on all runtimes</li>
<li><b>Uni-Dock (GPU):</b> 1000x+ speedup for large libraries</li>
</ul>

<p><em>MIT License | AcuDock Project</em></p>
</div>
""")

# === Assemble Tabs ===
tabs = widgets.Tab(children=[campaign_tab, about_tab])
tabs.set_title(0, 'Run Campaign')
tabs.set_title(1, 'About')

display(widgets.HTML('<h1>AcuDock Scout</h1>'
                     '<p><b>Active learning virtual screening</b> — '
                     'find top hits by docking &lt;10% of your library.</p>'))
display(tabs)
